# Notebook khung để huấn luyện mô hình trên Kaggle

Bố cục này giúp bạn code nhanh trên Kaggle, có sẵn các bước kiểm tra đường dẫn, đọc dữ liệu, huấn luyện và lưu model.

## 1. Thiết lập môi trường Kaggle và thư mục làm việc

Kiểm tra /kaggle/input, /kaggle/working. Tạo thư mục output và log nếu cần.

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")

print("input_root exists:", input_root.exists())
print("working_root exists:", working_root.exists())

output_dir = working_root / "outputs"
log_dir = working_root / "logs"
output_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

print("output_dir:", output_dir)
print("log_dir:", log_dir)


## 2. Nạp thư viện và cấu hình hạt giống

Import thư viện cần thiết và đặt seed để tái lập kết quả.

In [ ]:
import os
import random

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


set_seed(42)
print("seed set")


## 3. Tải và kiểm tra dữ liệu từ /kaggle/input

Đọc dữ liệu, thống kê cơ bản, hiển thị một vài mẫu.

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")
print("datasets in /kaggle/input:")
if input_root.exists():
    for p in input_root.iterdir():
        print("-", p.name)
else:
    print("/kaggle/input not found (not running on Kaggle)")

# TODO: update this to your dataset folder name
city_root = input_root / "cityscapes"
print("city_root exists:", city_root.exists())


## 4. Tiền xử lý và chia tập train/valid

Làm sạch, mã hóa, chuẩn hóa và chia dữ liệu thành train/validation.

In [ ]:
# TODO: implement preprocessing and split logic
# Example placeholders
train_images_dir = city_root / "train" / "images"
train_masks_dir = city_root / "train" / "masks"
val_images_dir = city_root / "val" / "images"
val_masks_dir = city_root / "val" / "masks"

print("train_images_dir:", train_images_dir)
print("val_images_dir:", val_images_dir)


## 5. Xây dựng mô hình baseline

Định nghĩa mô hình đơn giản phù hợp bài toán.

In [ ]:
import torch
from torch import nn


class SimpleUNet(nn.Module):
    def __init__(self, in_channels: int = 3, num_classes: int = 12) -> None:
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(inplace=True),
        )
        self.out = nn.Conv2d(32, num_classes, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.enc(x)
        return self.out(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleUNet().to(device)
print(model)


## 6. Huấn luyện và theo dõi metric

Viết vòng lặp huấn luyện, tính loss/metric và lưu history.

In [ ]:
from torch.utils.data import DataLoader

# TODO: build real Dataset and DataLoader
train_loader = []
val_loader = []

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

history = {"train_loss": [], "val_loss": []}

for epoch in range(1):
    model.train()
    running_loss = 0.0
    for batch in train_loader:
        images, masks = batch
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    history["train_loss"].append(running_loss)

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            images, masks = batch
            images = images.to(device)
            masks = masks.to(device)
            logits = model(images)
            loss = criterion(logits, masks)
            val_loss += loss.item()

    history["val_loss"].append(val_loss)

print("history:", history)


## 7. Đánh giá và lưu model

Đánh giá trên validation và lưu model vào /kaggle/working.

In [ ]:
ckpt_path = output_dir / "model.pt"
torch.save(model.state_dict(), ckpt_path)
print("saved:", ckpt_path)


## 8. Tạo file dự đoán và xuất submission

Sinh dự đoán cho test, tạo file submission CSV đúng định dạng.

In [ ]:
import pandas as pd

# TODO: replace with real predictions
sample_submission = pd.DataFrame({"id": [], "label": []})
sub_path = output_dir / "submission.csv"
sample_submission.to_csv(sub_path, index=False)
print("submission saved:", sub_path)
